In [11]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [12]:
from langchain_groq import ChatGroq
model = ChatGroq(
    model = "qwen/qwen3-32b"
)


In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str)->str:
    """ Get the weather at any location"""
    return f"The weather in {location} is sunny"

In [14]:
model_with_tools = model.bind_tools([get_weather])

In [15]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': '64md4j7zv', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.144785104, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.006165117, 'prompt_tokens_details': None, 'queue_time': 0.051376978, 'total_time': 0.150950221}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e6876-e0b4-7710-a

In [16]:
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops

In [18]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    print(tool_call)
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '0jqg91w5j', 'type': 'tool_call'}
The weather in Boston is currently sunny! If you're there, it's a great day to enjoy outdoor activities. 🌞


In [19]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': '0jqg91w5j', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.141903711, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.005934195, 'prompt_tokens_details': None, 'queue_time': 0.154933244, 'total_time': 0.147837906}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_